# An Illusion of Unlearning? — Full Experiment Notebook

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
(Gao, Unal, Rangamani, Zhu — AISTATS 2026)

This notebook reproduces **all experiments** from the paper:

| Stage | What runs |
|-------|----------|
| **A** | Environment setup & repo clone |
| **B** | Pre-train original model (ResNet-18 / CIFAR-10, configurable) |
| **C** | Retrain-on-retain baseline (gold standard) |
| **D** | Retain-only fine-tuning baseline |
| **E** | All 6 standard unlearning methods: NegGrad+, Random-Label, SalUn, SCRUB, UNSIR, SVD |
| **F** | CMF fine-tune (encoder prep) + all 5 CMF variants |
| **G** | Evaluation: Output acc, Linear Probe acc, NCC acc |
| **H** | Results table (Table 1 & Table 3 of paper) + bar chart |
| **I** | t-SNE visualisation |

> **Recommended:** GPU T4/P100 in **Settings → Accelerator**.  
> Full run (~300 epoch pre-train + all methods) takes ~4 h on T4.  
> Set `QUICK_MODE = True` below to run a fast demo (~20 min).

## A. Environment Setup

In [ ]:
import subprocess, sys

def sh(cmd, verbose=True):
    """Run shell command, print tail of output, return exit code."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout:
        print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr:
        print("STDERR:", r.stderr[-2000:])
    return r.returncode

sh("pip install -q timm einops scikit-learn matplotlib seaborn")

In [ ]:
import os, sys, json, copy, random, argparse, itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    print('Repo already present.')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## B. Configuration

Matches **Appendix A** of the paper exactly.  
Set `QUICK_MODE = True` for a fast smoke-test (fewer epochs, single forget class).  

**Method selection** is in the *"B2. Method Selection"* cell — comment/uncomment  
any combination of standard and CMF methods before running Section G.

In [ ]:
# ── Quick-mode toggle ─────────────────────────────────────────────────
QUICK_MODE = True    # False → full paper settings (~4 h on T4)

# ── Dataset / architecture (Appendix A.2, A.4) ───────────────────────
# ResNet experiments: resnet18 on cifar10/cifar100, resnet50 on tinyimagenet
# ViT experiments  : vit_s_16 on all three datasets (--pretrained)
DATASET  = 'cifar10'    # 'cifar10' | 'cifar100' | 'tinyimagenet'
ARCH     = 'resnet18'   # 'resnet18' | 'resnet50' | 'vit_s_16'
IS_VIT   = ARCH == 'vit_s_16'

DATA_PATH = '/kaggle/working/data'
SEED      = 1234

# ── Unlearning scenarios (Appendix A.3) ───────────────────────────────
# Each entry: list of class-indices to forget
if DATASET == 'cifar10':
    SINGLE_CLASS_EXPS = [[0],[1],[2],[3],[4],[5],[6],[7],[8],[9]]
    MULTI_CLASS_EXPS  = [[0,1,2],[3,4,5],[6,7,8],[0,5,9],[2,4,8]]
elif DATASET == 'cifar100':
    SINGLE_CLASS_EXPS = [[0],[1],[2],[3],[5]]
    MULTI_CLASS_EXPS  = [
        [3,15,19,21,31,38,42,43,88,97],
        [47,52,54,56,59,62,70,82,92,96],
        [5,20,22,25,39,40,84,86,87,94],
        [8,13,41,48,59,69,81,85,89,90],
        [1,4,30,32,55,67,72,73,91,95],
    ]
else:  # tinyimagenet
    SINGLE_CLASS_EXPS = [[2],[3],[5],[7],[9]]
    MULTI_CLASS_EXPS  = [
        list(range(0,20)), list(range(20,40)), list(range(40,60)),
        list(range(60,80)), list(range(80,100)),
    ]

# Quick-mode: run only the first single-class experiment
if QUICK_MODE:
    ALL_EXPS = [SINGLE_CLASS_EXPS[0]]
else:
    ALL_EXPS = SINGLE_CLASS_EXPS + MULTI_CLASS_EXPS

# ── Dataset size constants ────────────────────────────────────────────
_TOTAL     = {'cifar10':50000, 'cifar100':50000, 'tinyimagenet':100000}
_PER_CLASS = {'cifar10':5000,  'cifar100':500,   'tinyimagenet':500}

# ── Pre-training hyperparameters (Appendix A.4) ───────────────────────
# ResNet: SGD, lr=0.05, 300 epochs, warmup 5, cosine, patience 50
# ViT   : SGD, lr=3e-4, 10 epochs fine-tune from ImageNet weights
if IS_VIT:
    PRETRAIN_LR     = 3e-4 if DATASET != 'tinyimagenet' else 1e-4
    PRETRAIN_EPOCHS = 5 if QUICK_MODE else 10
else:
    PRETRAIN_LR     = 0.05
    PRETRAIN_EPOCHS = 20 if QUICK_MODE else 300
PRETRAIN_BS      = 128
PRETRAIN_PATIENCE = 10 if QUICK_MODE else 50

# ── Unlearning hyperparameters (Appendix A.6 / Tables 4-5) ───────────
# All LRs come from the shell scripts; reproduced here per method/dataset.
UNLEARN_BS = 128

# ResNet LR table  (method -> dataset -> single/multi)
_LR = {
    'random_label':              {'cifar10':{'s':1e-2,'m':1e-2}, 'cifar100':{'s':3e-3,'m':3e-3},  'tinyimagenet':{'s':5e-4,'m':5e-4}},
    'salun':                     {'cifar10':{'s':1e-2,'m':1e-2}, 'cifar100':{'s':3e-3,'m':3e-3},  'tinyimagenet':{'s':5e-4,'m':5e-4}},
    'grad_ascent_descent':       {'cifar10':{'s':1e-3,'m':1e-3}, 'cifar100':{'s':5e-5,'m':1e-4},  'tinyimagenet':{'s':5e-4,'m':5e-4}},
    'scrub':                     {'cifar10':{'s':1e-4,'m':1e-4}, 'cifar100':{'s':1e-3,'m':3e-4},  'tinyimagenet':{'s':5e-3,'m':1e-3}},
    'tarun':                     {'cifar10':{'s':2e-3,'m':5e-5}, 'cifar100':{'s':3e-5,'m':3e-5},  'tinyimagenet':{'s':2e-5,'m':2e-5}},
    'SVD':                       {'cifar10':{'s':1e-2,'m':1e-2}, 'cifar100':{'s':1e-2,'m':1e-2},  'tinyimagenet':{'s':1e-3,'m':1e-3}},
    'random_label_CMF_RemoveFC': {'cifar10':{'s':1e-4,'m':2e-3}, 'cifar100':{'s':2e-3,'m':2e-3},  'tinyimagenet':{'s':1e-2,'m':1e-2}},
    'salun_CMF_RemoveFC':        {'cifar10':{'s':2e-4,'m':2e-3}, 'cifar100':{'s':2e-3,'m':2e-3},  'tinyimagenet':{'s':1e-2,'m':1e-2}},
    'grad_ascent_descent_CMF_RemoveFC': {'cifar10':{'s':1e-4,'m':1e-4},'cifar100':{'s':1e-4,'m':1e-4},'tinyimagenet':{'s':3e-5,'m':3e-5}},
    'tarun_CMF_RemoveFC':        {'cifar10':{'s':5e-5,'m':5e-5}, 'cifar100':{'s':5e-5,'m':5e-5},  'tinyimagenet':{'s':2e-5,'m':2e-5}},
    'scrub_CMF_RemoveFC':        {'cifar10':{'s':5e-3,'m':1e-3}, 'cifar100':{'s':5e-3,'m':5e-3},  'tinyimagenet':{'s':5e-3,'m':1e-3}},
}

# Standard unlearning epochs (Appendix A.6)
_EPOCHS = {
    'random_label':3, 'salun':3, 'grad_ascent_descent':3,
    'scrub':3, 'tarun':3, 'SVD':200,
    'random_label_CMF_RemoveFC':4, 'salun_CMF_RemoveFC':4,
    'grad_ascent_descent_CMF_RemoveFC':3, 'tarun_CMF_RemoveFC':3,
    'scrub_CMF_RemoveFC':3,
}
if QUICK_MODE:
    _EPOCHS = {k: min(v, 2) for k, v in _EPOCHS.items()}

# SVD extra params (Appendix A.6)
_SVD = {
    'cifar10':    {'alpha_r':100,  'alpha_f':3,  'samples':900,  'max_patches':10000},
    'cifar100':   {'alpha_r':1000, 'alpha_f':30, 'samples':990,  'max_patches':10000},
    'tinyimagenet':{'alpha_r':30,  'alpha_f':10, 'samples':999,  'max_patches':10000},
}

def get_lr(method, forget_classes):
    mode = 's' if len(forget_classes)==1 else 'm'
    return _LR.get(method, {}).get(DATASET, {}).get(mode, 1e-3)

def get_epochs(method):
    return _EPOCHS.get(method, 3)

os.makedirs(DATA_PATH, exist_ok=True)
print(f'Dataset={DATASET}  Arch={ARCH}  VIT={IS_VIT}  QuickMode={QUICK_MODE}')
print(f'Experiments: {len(ALL_EXPS)} forget groups')

## B2. Method Selection

Choose which methods to run by editing `RUN_STANDARD` and `RUN_CMF`.  
Comment out any line to skip that method, or clear a list to skip the whole group.

| Key | Paper name | Group |
|-----|-----------|-------|
| `grad_ascent_descent` | NegGrad+ | Standard |
| `random_label` | Random Label | Standard |
| `salun` | SalUn | Standard |
| `scrub` | SCRUB | Standard |
| `tarun` | UNSIR | Standard |
| `SVD` | SVD (gradient-free) | Standard |
| `grad_ascent_descent_CMF_RemoveFC` | NegGrad+ w. CMF | CMF |
| `random_label_CMF_RemoveFC` | Random Label w. CMF | CMF |
| `salun_CMF_RemoveFC` | SalUn w. CMF | CMF |
| `scrub_CMF_RemoveFC` | SCRUB w. CMF | CMF |
| `tarun_CMF_RemoveFC` | UNSIR w. CMF | CMF |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CHOOSE WHICH METHODS TO RUN
#  Comment out any line to skip that method.
#  Set RUN_STANDARD = [] or RUN_CMF = [] to skip an entire group.
# ══════════════════════════════════════════════════════════════════════

# ── Standard methods (start from pre-trained checkpoint) ─────────────
RUN_STANDARD = [
    'grad_ascent_descent',          # NegGrad+
    'random_label',                 # Random Label
    'salun',                        # SalUn
    'scrub',                        # SCRUB
    'tarun',                        # UNSIR
    'SVD',                          # SVD  (gradient-free)
]

# ── CMF methods (start from CMF_FT_RemoveFC checkpoint) ──────────────
# Section F must have run first to build that checkpoint.
RUN_CMF = [
    'grad_ascent_descent_CMF_RemoveFC',   # NegGrad+ w. CMF
    'random_label_CMF_RemoveFC',          # Random Label w. CMF
    'salun_CMF_RemoveFC',                 # SalUn w. CMF
    'scrub_CMF_RemoveFC',                 # SCRUB w. CMF
    'tarun_CMF_RemoveFC',                 # UNSIR w. CMF
]

# ── Validation ───────────────────────────────────────────────────────
_ALL_KNOWN = {
    'grad_ascent_descent', 'random_label', 'salun', 'scrub', 'tarun', 'SVD',
    'grad_ascent_descent_CMF_RemoveFC', 'random_label_CMF_RemoveFC',
    'salun_CMF_RemoveFC', 'scrub_CMF_RemoveFC', 'tarun_CMF_RemoveFC',
}
for _m in RUN_STANDARD + RUN_CMF:
    assert _m in _ALL_KNOWN, f'Unknown method: {_m!r}'

# Deduplicate while preserving order
RUN_STANDARD = list(dict.fromkeys(RUN_STANDARD))
RUN_CMF      = list(dict.fromkeys(RUN_CMF))

total_runs = (len(RUN_STANDARD) + len(RUN_CMF)) * len(ALL_EXPS)
print(f'Standard methods selected ({len(RUN_STANDARD)}): {RUN_STANDARD}')
print(f'CMF methods selected     ({len(RUN_CMF)}): {RUN_CMF}')
print(f'Total: {len(RUN_STANDARD)+len(RUN_CMF)} methods x {len(ALL_EXPS)} forget groups = {total_runs} runs')

## C. Args Helper

In [ ]:
from utils import get_dataset, get_model, get_retain_forget_partition, test, load_encoder_ckpt_safely
from unlearn import unlear_func
from train import train as train_one_epoch

def make_args(**ov):
    """Build argparse.Namespace matching main.py defaults, overridden by ov."""
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=None, class_label_names=None,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=SEED, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5,
        min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET],
        num_forget_samples=0,
        grad_norm_clip=None, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64,
        scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3,
        SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=False, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='paper_repro',
    )
    d.update(ov)
    return argparse.Namespace(**d)

print('Helpers loaded.')

## D. Load Dataset

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
NUM_CLASSES       = args_base.num_classes
CLASS_LABEL_NAMES = args_base.class_label_names
print(f'Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW  = dict(batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True, shuffle=True)
TEST_KW    = dict(batch_size=256,         num_workers=2, pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(dataset_train, **LOADER_KW)
test_loader  = torch.utils.data.DataLoader(dataset_test,  **TEST_KW)

## E. Pre-train Original Model (Appendix A.4)

**ResNet:** SGD lr=0.05, batch 128, ≤300 epochs, warmup 5, cosine decay, patience 50.  
**ViT-S/16:** Fine-tune ImageNet weights, lr=3×10⁻⁴, 10 epochs.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR

CKPT_PRETRAIN = f'{REPO_DIR}/checkpoints/pre_train/{DATASET}_{ARCH}.pt'
os.makedirs(os.path.dirname(CKPT_PRETRAIN), exist_ok=True)

args_pt = make_args(unlearn_method='pre_train',
                    epochs_or_steps=PRETRAIN_EPOCHS,
                    lr=PRETRAIN_LR,
                    num_classes=NUM_CLASSES,
                    class_label_names=CLASS_LABEL_NAMES,
                    patience=PRETRAIN_PATIENCE)

orig_model = get_model(args_pt, device)

if os.path.exists(CKPT_PRETRAIN):
    print(f'Loading existing checkpoint: {CKPT_PRETRAIN}')
    orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
else:
    print(f'Pre-training {ARCH} on {DATASET} …')
    total_len = len(dataset_train)
    val_len   = int(total_len * 0.1)
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(total_len, generator=g).tolist()
    tr_loader_pt = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset_train, idx[:total_len-val_len]), **LOADER_KW)
    va_loader_pt = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset_train, idx[total_len-val_len:]), **TEST_KW)

    optimizer = optim.SGD(orig_model.parameters(), lr=PRETRAIN_LR,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)
    warmup_sched = LambdaLR(optimizer, lr_lambda=lambda e: min(1.0, (e+1)/5))
    cosine_sched = CosineAnnealingLR(optimizer,
                                     T_max=max(1, PRETRAIN_EPOCHS-5), eta_min=1e-5)
    scheduler = SequentialLR(optimizer,
                              schedulers=[warmup_sched, cosine_sched],
                              milestones=[5])

    best_acc, no_impr = 0.0, 0
    for epoch in range(1, PRETRAIN_EPOCHS+1):
        train_one_epoch(args_pt, orig_model, device, tr_loader_pt, optimizer, epoch)
        va, _, _ = test(orig_model, device, va_loader_pt, [], CLASS_LABEL_NAMES,
                        NUM_CLASSES, plot_cm=False, job_name='pretrain', set_name='Val')
        scheduler.step()
        if va > best_acc:
            best_acc, no_impr = va, 0
            torch.save(orig_model.state_dict(), CKPT_PRETRAIN)
            print(f'  epoch {epoch:3d}: val={va:.4f} ✓ saved')
        else:
            no_impr += 1
            if no_impr >= PRETRAIN_PATIENCE:
                print(f'  Early stop at epoch {epoch}')
                break
    orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))

orig_model.eval()
print('\n── Original model ──')
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES,
     plot_cm=False, job_name='original', set_name='Test')

## F. CMF Fine-Tune — Prepare CMF Encoder (Appendix B)

Fine-tune the original encoder with the CMF head on the **full** dataset (no forget/retain split).  
This checkpoint is the starting point for all CMF-based unlearning methods.

In [ ]:
CMF_FT_LR     = PRETRAIN_LR if IS_VIT else 1e-3
CMF_FT_EPOCHS = 2 if QUICK_MODE else 1   # paper: 1 epoch is enough

CKPT_CMF_FT = f'{REPO_DIR}/checkpoints/CMF_FT_RemoveFC/{DATASET}_{ARCH}.pt'
os.makedirs(os.path.dirname(CKPT_CMF_FT), exist_ok=True)

args_cmf_ft = make_args(
    unlearn_method='CMF_FT_RemoveFC',
    epochs_or_steps=CMF_FT_EPOCHS,
    lr=CMF_FT_LR,
    num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
    num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
    unlearn_class=[], remove_FC=True, CMFClassifier=True,
    weight_decay=1e-4,
)

if os.path.exists(CKPT_CMF_FT):
    print('CMF_FT checkpoint already exists.')
else:
    cmf_base = get_model(args_cmf_ft, device)
    print('Loading pretrained weights into CMF model …')
    load_encoder_ckpt_safely(cmf_base, CKPT_PRETRAIN, device=str(device))
    opt_ft = optim.SGD(cmf_base.parameters(), lr=CMF_FT_LR,
                       momentum=0.9, weight_decay=1e-4, nesterov=True)
    print('Running CMF_FT_RemoveFC …')
    cmf_ft_model = unlear_func['CMF_FT_RemoveFC'](
        args=args_cmf_ft, model=cmf_base, device=device,
        retain_loader=train_loader, forget_loader=None,
        train_loader=train_loader, val_loader=None,
        test_loader=test_loader, optimizer=opt_ft,
        epochs=CMF_FT_EPOCHS, train_dataset=dataset_train,
        val_index=np.arange(len(dataset_train)),
        test_forget_loader=torch.utils.data.DataLoader(dataset_test, **TEST_KW),
    )
    torch.save(cmf_ft_model.state_dict(), CKPT_CMF_FT)
    print('Saved:', CKPT_CMF_FT)

print('CMF_FT_RemoveFC base checkpoint ready.')

## G. Run All Unlearning Methods

Iterates over all forget-class experiments × all methods.  
Results are accumulated in `RESULTS` for the summary table.

In [ ]:
# ── Shared run helper ─────────────────────────────────────────────────
def run_unlearn(method, forget_classes, start_ckpt, extra_args=None):
    """
    Run one unlearning experiment.
    Returns (retain_acc, forget_acc, model).
    """
    n_forget = len(forget_classes)
    num_forget = n_forget * _PER_CLASS[DATASET]
    num_retain = _TOTAL[DATASET] - num_forget
    forget_str = ','.join(str(c) for c in forget_classes)

    lr     = get_lr(method, forget_classes)
    epochs = get_epochs(method)
    is_cmf = 'CMF' in method

    kw = dict(
        unlearn_method=method,
        epochs_or_steps=epochs,
        lr=lr, batch_size=UNLEARN_BS,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        num_retain_samples=num_retain, num_forget_samples=num_forget,
        unlearn_class=list(forget_classes),
        remove_FC=is_cmf, CMFClassifier=is_cmf,
        grad_norm_clip=1.0,
        lp_every=0, ncc_every=0,
    )
    if method == 'salun' or method == 'salun_CMF_RemoveFC':
        kw['salun_threshold'] = 0.5
    if method == 'SVD':
        s = _SVD[DATASET]
        kw.update(SVD_alpha_r=s['alpha_r'], SVD_alpha_f=s['alpha_f'],
                  SVD_samples=s['samples'], SVD_max_patches=s['max_patches'])
    if method in ('tarun','tarun_CMF_RemoveFC'):
        kw.update(tarun_impair_lr=2e-4 if DATASET!='cifar10' else 1e-4,
                  tarun_samples_per_class=1000)
    if method in ('scrub','scrub_CMF_RemoveFC'):
        kw.update(scrub_del_bsz=64, scrub_sgda_bsz=64,
                  scrub_msteps=2, scrub_epochs=epochs)
    if extra_args:
        kw.update(extra_args)

    args = make_args(**kw)

    # prepare retain / forget loaders
    retain_ds, forget_ds = get_retain_forget_partition(args, dataset_train, forget_classes)
    retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)
    forget_loader = (torch.utils.data.DataLoader(forget_ds, **LOADER_KW)
                     if len(forget_ds) > 0 else None)
    _, test_forget_ds = get_retain_forget_partition(args, dataset_test, forget_classes)
    tf_loader = torch.utils.data.DataLoader(test_forget_ds, **TEST_KW)

    # initialise model
    m = get_model(args, device)
    state = torch.load(start_ckpt, map_location=device)
    m.load_state_dict(state)

    optimizer = optim.SGD(m.parameters(), lr=lr,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)

    print(f'  [{method}] forget={forget_classes} lr={lr} epochs={epochs}')
    unlearnt = unlear_func[method](
        args=args, model=m, device=device,
        retain_loader=retain_loader,
        forget_loader=forget_loader,
        train_loader=train_loader,
        val_loader=None, test_loader=test_loader,
        optimizer=optimizer, epochs=epochs,
        train_dataset=dataset_train,
        val_index=np.arange(len(dataset_train)),
        test_forget_loader=tf_loader,
    )

    # save checkpoint
    ckpt_out = (f'{REPO_DIR}/checkpoints/{method}/{DATASET}_{ARCH}/{forget_str}.pt')
    os.makedirs(os.path.dirname(ckpt_out), exist_ok=True)
    torch.save(unlearnt.state_dict(), ckpt_out)

    # output-level accuracy
    unlearnt.eval()
    ra, fa, _ = test(unlearnt, device, test_loader, forget_classes,
                     CLASS_LABEL_NAMES, NUM_CLASSES,
                     plot_cm=False, job_name=method, set_name='Test')
    return ra, fa, unlearnt

print('run_unlearn helper defined.')

In [ ]:
# ── Gold-standard: Retrain on retain set (Appendix A.5) ───────────────
# ResNet: same training settings as pre-train but on retain data only.
# ViT   : fine-tune from ImageNet weights, 10 epochs, lr=3e-4.

RESULTS = []  # accumulated across all experiments

for forget_classes in ALL_EXPS:
    n_forget   = len(forget_classes)
    num_forget = n_forget * _PER_CLASS[DATASET]
    num_retain = _TOTAL[DATASET] - num_forget
    forget_str = ','.join(str(c) for c in forget_classes)
    mode       = 'single' if n_forget==1 else 'multi'

    print(f'\n{'='*60}')
    print(f'Forget classes: {forget_classes}  ({mode})')
    print('='*60)

    # Retain-only Retrain
    rt_ckpt = f'{REPO_DIR}/checkpoints/retrain/{DATASET}_{ARCH}/{forget_str}.pt'
    os.makedirs(os.path.dirname(rt_ckpt), exist_ok=True)
    if not os.path.exists(rt_ckpt):
        retain_ds, _ = get_retain_forget_partition(
            make_args(num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
                      unlearn_class=list(forget_classes)),
            dataset_train, forget_classes)
        rt_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)

        rt_args = make_args(
            unlearn_method='retrain',
            epochs_or_steps=PRETRAIN_EPOCHS, lr=PRETRAIN_LR,
            num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
            num_retain_samples=num_retain, num_forget_samples=num_forget,
            unlearn_class=list(forget_classes),
        )
        rt_model = get_model(rt_args, device)
        rt_opt   = optim.SGD(rt_model.parameters(), lr=PRETRAIN_LR,
                             momentum=0.9, weight_decay=5e-4, nesterov=True)
        _, __, _ = run_unlearn.__wrapped__  if hasattr(run_unlearn,'__wrapped__') else (None,None,None)
        # run retrain via unlear_func
        print('  [retrain]', forget_classes)
        rt_model = unlear_func['retrain'](
            args=rt_args, model=rt_model, device=device,
            retain_loader=rt_loader, forget_loader=None,
            train_loader=rt_loader, val_loader=None,
            test_loader=test_loader, optimizer=rt_opt,
            epochs=PRETRAIN_EPOCHS, train_dataset=dataset_train,
            val_index=np.arange(len(dataset_train)),
            test_forget_loader=torch.utils.data.DataLoader(dataset_test, **TEST_KW),
        )
        torch.save(rt_model.state_dict(), rt_ckpt)
    else:
        rt_args = make_args(num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
                            unlearn_class=list(forget_classes))
        rt_model = get_model(rt_args, device)
        rt_model.load_state_dict(torch.load(rt_ckpt, map_location=device))

    rt_model.eval()
    rt_ra, rt_fa, _ = test(rt_model, device, test_loader, forget_classes,
                           CLASS_LABEL_NAMES, NUM_CLASSES,
                           plot_cm=False, job_name='retrain', set_name='Test')
    RESULTS.append(dict(forget=forget_str, method='Retrain',
                        retain_acc=rt_ra, forget_acc=rt_fa))

print('\nRetrain baselines done.')

In [ ]:
# ── Standard unlearning methods (Table 1 of paper) ────────────────────
# Uses RUN_STANDARD defined in the Method Selection cell above.
STANDARD_METHODS = RUN_STANDARD

if not STANDARD_METHODS:
    print('No standard methods selected — skipping.')
else:
    print(f'Running {len(STANDARD_METHODS)} standard method(s): {STANDARD_METHODS}')

for forget_classes in ALL_EXPS:
    forget_str = ','.join(str(c) for c in forget_classes)
    if not STANDARD_METHODS:
        break
    print(f'\n>>> forget={forget_classes}')
    for method in STANDARD_METHODS:
        try:
            ra, fa, _ = run_unlearn(method, forget_classes, CKPT_PRETRAIN)
            RESULTS.append(dict(forget=forget_str, method=method,
                                retain_acc=ra, forget_acc=fa))
        except Exception as e:
            print(f'  ERROR {method}: {e}')
            RESULTS.append(dict(forget=forget_str, method=method,
                                retain_acc=float('nan'), forget_acc=float('nan')))

print('\nStandard methods done.')

In [ ]:
# ── CMF-enhanced unlearning methods (Table 3 of paper) ────────────────
# Uses RUN_CMF defined in the Method Selection cell above.
CMF_METHODS = RUN_CMF

if not CMF_METHODS:
    print('No CMF methods selected — skipping.')
else:
    print(f'Running {len(CMF_METHODS)} CMF method(s): {CMF_METHODS}')

for forget_classes in ALL_EXPS:
    forget_str = ','.join(str(c) for c in forget_classes)
    if not CMF_METHODS:
        break
    print(f'\n>>> forget={forget_classes}')
    for method in CMF_METHODS:
        try:
            ra, fa, _ = run_unlearn(method, forget_classes, CKPT_CMF_FT)
            RESULTS.append(dict(forget=forget_str, method=method,
                                retain_acc=ra, forget_acc=fa))
        except Exception as e:
            print(f'  ERROR {method}: {e}')
            RESULTS.append(dict(forget=forget_str, method=method,
                                retain_acc=float('nan'), forget_acc=float('nan')))

print('\nCMF methods done.')

## H. Evaluation — Output, Linear Probe, NCC  (Table 1 & 3)

For each saved checkpoint we run `evaluation.py` which computes:
- **Output** accuracy (forget & retain)
- **Linear Probe** accuracy (frozen features + new linear head)
- **NCC** accuracy (nearest-class-centre classifier)

In [ ]:
# Run evaluation.py for each (method, forget_group) that has a checkpoint.
# Results are written to evaluations/<method>/<dataset>_<arch>/<classes>.log

ALL_METHODS = list(dict.fromkeys(STANDARD_METHODS + CMF_METHODS))  # preserve order, no dups
EVAL_RESULTS = []  # rows: {method, forget, eval_type, retain_acc, forget_acc}

for forget_classes in ALL_EXPS:
    n_forget   = len(forget_classes)
    num_forget = n_forget * _PER_CLASS[DATASET]
    num_retain = _TOTAL[DATASET] - num_forget
    forget_str = ','.join(str(c) for c in forget_classes)
    is_cmf_exp = False

    for method in ALL_METHODS:
        ckpt = f'{REPO_DIR}/checkpoints/{method}/{DATASET}_{ARCH}/{forget_str}.pt'
        if not os.path.exists(ckpt):
            continue
        is_cmf_exp = 'CMF' in method
        eval_log_dir = f'{REPO_DIR}/evaluations/{method}/{DATASET}_{ARCH}'
        os.makedirs(eval_log_dir, exist_ok=True)
        logfile = f'{eval_log_dir}/{forget_str.replace(",","_")}.log'

        cmd = (
            f'python {REPO_DIR}/evaluation.py'
            f' --dataset {DATASET}'
            f' --arch {ARCH}'
            f' --unlearn-method {method}'
            f' --epochs-or-steps 10'
            f' --batch-size 128'
            f' --lr 3e-4'
            f' --num-retain-samples {num_retain}'
            f' --num-forget-samples {num_forget}'
            f' --unlearn-class "{forget_str}"'
            f' --prob-batch-size 128'
            f' --do-linear-probe'
            f' --use-last-only'
            f' --do-ncc-mismatch'
            + (' --remove_FC' if is_cmf_exp else '')
            + (' --pretrained' if IS_VIT else '')
            + f' > {logfile} 2>&1'
        )
        print(f'Eval {method} forget={forget_classes} …')
        rc = sh(cmd, verbose=False)
        if rc != 0:
            print(f'  WARNING: eval exited with {rc}, check {logfile}')
        else:
            # parse last lines of log
            with open(logfile) as f:
                lines = f.read()
            print(lines[-600:] if len(lines)>600 else lines)

print('Evaluation complete.')

## I. Results Summary Table (Table 1 & 3 of paper)

In [ ]:
# Output-level results accumulated in RESULTS
df = pd.DataFrame(RESULTS)

# Add Original model row
orig_model.eval()
orig_rows = []
for forget_classes in ALL_EXPS:
    forget_str = ','.join(str(c) for c in forget_classes)
    ra, fa, _ = test(orig_model, device, test_loader, forget_classes,
                     CLASS_LABEL_NAMES, NUM_CLASSES,
                     plot_cm=False, job_name='original', set_name='Test')
    orig_rows.append(dict(forget=forget_str, method='Original',
                          retain_acc=ra, forget_acc=fa))
df = pd.concat([pd.DataFrame(orig_rows), df], ignore_index=True)

# Pivot: mean across forget groups
summary = df.groupby('method')[['retain_acc','forget_acc']].mean()
summary.columns = ['Mean Retain Acc', 'Mean Forget Acc']
summary = summary.round(4)

# Order rows to match paper
ROW_ORDER = [
    'Original', 'Retrain',
    'grad_ascent_descent', 'random_label', 'salun', 'scrub', 'tarun', 'SVD',
    'random_label_CMF_RemoveFC', 'salun_CMF_RemoveFC',
    'grad_ascent_descent_CMF_RemoveFC', 'scrub_CMF_RemoveFC', 'tarun_CMF_RemoveFC',
]
present = [r for r in ROW_ORDER if r in summary.index]
summary = summary.loc[present]

# Pretty labels matching paper
NAME_MAP = {
    'Original': 'Original',
    'Retrain': 'Retain-only Retrain',
    'grad_ascent_descent': 'NegGrad+',
    'random_label': 'Random Label',
    'salun': 'SalUn',
    'scrub': 'SCRUB',
    'tarun': 'UNSIR',
    'SVD': 'SVD',
    'random_label_CMF_RemoveFC': 'Random Label w. CMF',
    'salun_CMF_RemoveFC': 'SalUn w. CMF',
    'grad_ascent_descent_CMF_RemoveFC': 'NegGrad+ w. CMF',
    'scrub_CMF_RemoveFC': 'SCRUB w. CMF',
    'tarun_CMF_RemoveFC': 'UNSIR w. CMF',
}
summary.index = [NAME_MAP.get(i, i) for i in summary.index]

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 30)
print(f'\n=== Output-Level Results — {DATASET} / {ARCH} (averaged over {len(ALL_EXPS)} forget groups) ===')
print(summary.to_string())

# Save to CSV
csv_path = f'{REPO_DIR}/results_{DATASET}_{ARCH}.csv'
df.to_csv(csv_path, index=False)
print(f'\nFull results saved: {csv_path}')

## J. Visualisation — Retain vs Forget Accuracy (Figure 1 style)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

labels = list(summary.index)
x = np.arange(len(labels))
w = 0.38

b1 = ax.bar(x - w/2, summary['Mean Retain Acc'], w,
            label='Retain Acc', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w/2, summary['Mean Forget Acc'], w,
            label='Forget Acc', color='tomato', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.08)
ax.set_title(f'Output-Level Accuracy — {DATASET} / {ARCH}\n'
             f'(mean over forget groups, Output level)')
ax.legend()
ax.axvline(x=1.5, color='grey', linewidth=0.8, linestyle='--')
ax.axvline(x=7.5, color='grey', linewidth=0.8, linestyle='--')
ax.text(0.75, 1.03, 'Baselines', ha='center', fontsize=8, transform=ax.transData)
ax.text(4.75, 1.03, 'Standard Methods', ha='center', fontsize=8, transform=ax.transData)
ax.text(10.0, 1.03, 'CMF Methods', ha='center', fontsize=8, transform=ax.transData)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=6.5)

plt.tight_layout()
fig_path = f'{REPO_DIR}/results_chart_{DATASET}_{ARCH}.png'
plt.savefig(fig_path, dpi=130)
plt.show()
print('Chart saved:', fig_path)

## K. t-SNE Visualisation (Figure 3 & 6 of paper)

Compares feature-space distributions for the Original model vs. a selected unlearning method.  
The forgotten class should be **linearly separable** in the standard method (Figure 3)  
and **merged with retain classes** in the CMF method (Figure 6).

In [ ]:
from sklearn.manifold import TSNE

def collect_features(model, loader, max_pts=3000):
    """Extract penultimate-layer features and labels."""
    model.eval()
    feats, labs = [], []
    seen = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            # get features before the final linear layer
            if hasattr(model, 'encoder'):
                z = model.encoder(x)
            elif hasattr(model, 'feature'):
                z = model.feature(x)
            else:
                # hook on avgpool / penultimate
                hooks, buf = [], []
                def _hook(mod, inp, out):
                    buf.append(out.detach().cpu().flatten(1))
                for name, mod in reversed(list(model.named_modules())):
                    if isinstance(mod, (nn.AdaptiveAvgPool2d, nn.LayerNorm)):
                        hooks.append(mod.register_forward_hook(_hook))
                        break
                model(x)
                for h in hooks: h.remove()
                z = buf[0] if buf else model(x).detach().cpu()
            feats.append(z.cpu().numpy() if isinstance(z, torch.Tensor) else z)
            labs.append(y.numpy())
            seen += x.size(0)
            if seen >= max_pts:
                break
    return np.concatenate(feats)[:max_pts], np.concatenate(labs)[:max_pts]


def plot_tsne(model, loader, title, forget_cls, max_pts=2000, ax=None):
    feats, labs = collect_features(model, loader, max_pts)
    emb = TSNE(n_components=2, perplexity=30, random_state=0,
               n_iter=500).fit_transform(feats)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    for c in sorted(set(labs)):
        mask = labs == c
        color = 'red' if c in forget_cls else None
        zorder = 5 if c in forget_cls else 1
        ax.scatter(emb[mask, 0], emb[mask, 1], s=6, alpha=0.5,
                   color=color, zorder=zorder, label=f'cls {c}' if c in forget_cls else None)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    return ax


# Pick first experiment group for visualisation
vis_forget = ALL_EXPS[0]
vis_forget_str = ','.join(str(c) for c in vis_forget)

# Build a small test loader for visualisation
vis_loader = torch.utils.data.DataLoader(dataset_test, batch_size=256,
                                          shuffle=True, num_workers=2)

# Models to visualise: Original + random_label + random_label_CMF_RemoveFC
vis_models = [('Original', CKPT_PRETRAIN, False)]
for mth in ['random_label', 'random_label_CMF_RemoveFC']:
    ck = f'{REPO_DIR}/checkpoints/{mth}/{DATASET}_{ARCH}/{vis_forget_str}.pt'
    if os.path.exists(ck):
        vis_models.append((mth, ck, 'CMF' in mth))

fig, axes = plt.subplots(1, len(vis_models), figsize=(5*len(vis_models), 4.5))
if len(vis_models) == 1:
    axes = [axes]

for ax, (name, ck, is_cmf) in zip(axes, vis_models):
    a_pt = make_args(num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
                     unlearn_class=list(vis_forget), remove_FC=is_cmf)
    m = get_model(a_pt, device)
    m.load_state_dict(torch.load(ck, map_location=device))
    m.eval()
    label = name.replace('_CMF_RemoveFC', ' w. CMF').replace('_', ' ')
    plot_tsne(m, vis_loader, label, vis_forget, max_pts=2000, ax=ax)
    ax.legend(loc='lower right', fontsize=7, markerscale=2)

plt.suptitle(f't-SNE — {DATASET}/{ARCH} | forget class(es): {vis_forget}  (red = forgotten)',
             fontsize=11)
plt.tight_layout()
tsne_path = f'{REPO_DIR}/tsne_{DATASET}_{ARCH}.png'
plt.savefig(tsne_path, dpi=130)
plt.show()
print('t-SNE saved:', tsne_path)

## L. Run All Experiments via Original Shell Scripts (Alternative)

If you prefer to use the original bash scripts unchanged, uncomment below.  
Each script handles multi-GPU fan-out; on Kaggle single-GPU they run sequentially.

In [ ]:
# ── (A) Pre-train ResNet ──────────────────────────────────────────────
# Uncomment ONE of the following:

# ResNet on CIFAR-10 / CIFAR-100:
# sh(f'bash {REPO_DIR}/script/run/run_resnet_unlearning.sh {DATA_PATH}')

# ResNet retrain baselines:
# sh(f'bash {REPO_DIR}/script/run/run_resnet_retrain.sh {DATA_PATH}')

# ── (B) CMF Unlearning — ResNet ───────────────────────────────────────
# sh(f'bash {REPO_DIR}/script/run/run_resnet_18_CMF_unlearning.sh {DATA_PATH}')

# ── (C) ViT fine-tune (pre-train from ImageNet weights) ───────────────
# sh(f'bash {REPO_DIR}/script/run/run_vit_finetune.sh {DATA_PATH}')

# ── (D) ViT retrain baselines ─────────────────────────────────────────
# sh(f'bash {REPO_DIR}/script/run/run_vit_retrain_finetune.sh {DATA_PATH}')

# ── (E) ViT standard unlearning ───────────────────────────────────────
# sh(f'bash {REPO_DIR}/script/run/run_vit_s16_unlearning.sh {DATA_PATH}')

# ── (F) ViT CMF unlearning ────────────────────────────────────────────
# sh(f'bash {REPO_DIR}/script/run/run_vit_s16_CMF_unlearning.sh {DATA_PATH}')

# ── (G) Evaluation — ResNet ───────────────────────────────────────────
# sh(f'bash {REPO_DIR}/script/eval/eval_resnet_unlearning.sh')
# sh(f'bash {REPO_DIR}/script/eval/eval_resnet_CMF_unlearning.sh')

# ── (H) Evaluation — ViT ─────────────────────────────────────────────
# sh(f'bash {REPO_DIR}/script/eval/eval_vit_unlearn.sh')
# sh(f'bash {REPO_DIR}/script/eval/eval_vit_CMF_unlearn.sh')

print('Uncomment any script block above to run it.')

---
## Summary

| Stage | Description | Paper reference |
|-------|-------------|----------------|
| Original model | Trained from scratch (ResNet) or fine-tuned from ImageNet (ViT) | Appendix A.4 |
| Retrain baseline | Retrain on retain data only — gold standard | Appendix A.5 |
| Standard methods | NegGrad+, Random-Label, SalUn, SCRUB, UNSIR, SVD | Table 1 |
| CMF fine-tune | Encoder prep: set CMF head via class-mean features | Algorithm 1 |
| CMF methods | Same 5 methods but with CMF classifier alignment | Table 3 |
| Evaluation | Output acc · Linear Probe acc · NCC acc | Section 3 |
| t-SNE | Visualise feature space; forgotten class separability | Figure 3 & 6 |

**Key finding (paper):**  
Standard methods achieve near-zero *output-level* forget accuracy but features remain  
linearly separable (high *linear probe* forget accuracy) — an *illusion of unlearning*.  
CMF methods break this illusion by enforcing feature–classifier alignment.